In [ ]:
%pip install scikit-opt

In [ ]:
import torch
import numpy as np
import math
import time
from sko.GA import GA

# ==========================================
# ⚙️ 1. 系统参数 — 严格对齐 MATLAB search_channel.m
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

room_x, room_y, room_z_max = 10.0, 10.0, 3.0
x_BS, y_BS, z_BS = 10.0, -100.0, -10.0

E = 8; d_B = 0.075; lambda_val = 0.075; L1 = 2
d_ref = abs(y_BS) * 1.5; P_BS_dBm = 40.0; R_th = 0.2
N0_dBm_Hz = -174.0; B = 20e6; p_m = 1.0 / 5.0; n_users = 5

P_BS = 10**(P_BS_dBm / 10.0) * 1e-3
N0 = 10**(N0_dBm_Hz / 10.0) * 1e-3 * B

# 轨迹参数 — 对齐 search_channel.m (sim_time=300, dt=10)
SIM_TIME = 300
DT = 10
TIME_STEPS = int(SIM_TIME / DT)      # 30
N_TOTAL_POINTS = n_users * TIME_STEPS  # 150

# ==========================================
# 🚶 2. 人类轨迹生成器 — 完整移植 MATLAB getHumanPosi_custom
# ==========================================
def generate_human_trajectory(n_users=n_users, room_x=room_x, room_y=room_y,
                               sim_time=SIM_TIME, dt=DT, rng=None):
    """
    完整移植 MATLAB search_channel.m 底部的 getHumanPosi_custom
    返回: total_points [n_users*time_steps, 3]  (x, y, z)
    """
    if rng is None:
        rng = np.random

    room_size = [0.0, room_x, 0.0, room_y]
    hotspot_center = np.array([room_x / 2, room_y / 2])
    hotspot_radius = room_x / 3.0

    p_t = 0.6; p_s = 0.3; tau_h = 1.5; tau_n = 0.1
    v_h = 0.2; v_n = 1.0; g_h = 0.6; g_n = 0.2

    time_steps = int(sim_time / dt)

    def generate_target(current_pos):
        """对齐 MATLAB generate_target_pos_custom"""
        if rng.rand() < g_h:
            target = hotspot_center + (rng.rand(2) - 0.5) * 2 * hotspot_radius
        else:
            target = np.array([
                room_size[0] + rng.rand() * (room_size[1] - room_size[0]),
                room_size[2] + rng.rand() * (room_size[3] - room_size[2])
            ])
        target[0] = np.clip(target[0], room_size[0], room_size[1])
        target[1] = np.clip(target[1], room_size[2], room_size[3])
        return target

    def get_tau(region):
        return tau_h if region == 'hot' else tau_n

    def get_speed(region):
        return v_h if region == 'hot' else v_n

    # 用户高度（随机但固定）
    users_height = 1.5 + 0.5 * rng.rand(n_users)

    positions = np.zeros((n_users, time_steps, 2))
    user_pos = np.zeros((n_users, 2))
    user_region = [None] * n_users
    user_state = [None] * n_users
    user_target = np.zeros((n_users, 2))
    user_dir = np.zeros((n_users, 2))
    user_speed = np.zeros(n_users)
    user_pause_remain = np.zeros(n_users)

    for i in range(n_users):
        user_pos[i, 0] = room_size[0] + rng.rand() * (room_size[1] - room_size[0])
        user_pos[i, 1] = room_size[2] + rng.rand() * (room_size[3] - room_size[2])
        dist2hot = np.linalg.norm(user_pos[i] - hotspot_center)
        user_region[i] = 'hot' if dist2hot <= hotspot_radius else 'normal'

        if rng.rand() < p_t:
            user_state[i] = 'transfer'
            user_target[i] = generate_target(user_pos[i])
            dir_vec = user_target[i] - user_pos[i]
            user_dir[i] = dir_vec / np.linalg.norm(dir_vec)
            user_speed[i] = get_speed(user_region[i])
        else:
            user_state[i] = 'pause'
            user_pause_remain[i] = rng.exponential(get_tau(user_region[i]))

        positions[i, 0, :] = user_pos[i]

    for step in range(1, time_steps):
        for i in range(n_users):
            if user_state[i] == 'pause':
                user_pause_remain[i] -= dt
                positions[i, step, :] = user_pos[i]
                if user_pause_remain[i] <= 0:
                    user_state[i] = 'transfer'
                    user_target[i] = generate_target(user_pos[i])
                    dir_vec = user_target[i] - user_pos[i]
                    user_dir[i] = dir_vec / np.linalg.norm(dir_vec)
                    user_speed[i] = get_speed(user_region[i])
            else:  # transfer
                move_dist = user_speed[i] * dt
                pos_diff = user_target[i] - user_pos[i]
                if np.linalg.norm(pos_diff) <= move_dist:
                    user_pos[i] = user_target[i].copy()
                    dist2hot = np.linalg.norm(user_pos[i] - hotspot_center)
                    user_region[i] = 'hot' if dist2hot <= hotspot_radius else 'normal'
                    if rng.rand() < p_s:
                        user_state[i] = 'pause'
                        user_pause_remain[i] = rng.exponential(get_tau(user_region[i]))
                    else:
                        user_target[i] = generate_target(user_pos[i])
                        dir_vec = user_target[i] - user_pos[i]
                        user_dir[i] = dir_vec / np.linalg.norm(dir_vec)
                        user_speed[i] = get_speed(user_region[i])
                else:
                    user_pos[i] = user_pos[i] + user_dir[i] * move_dist
                positions[i, step, :] = user_pos[i]

    # 展平为 [n_users*time_steps, 3]
    total_points = np.zeros((n_users * time_steps, 3))
    idx = 0
    for user in range(n_users):
        for step in range(time_steps):
            total_points[idx, 0] = positions[user, step, 0]
            total_points[idx, 1] = positions[user, step, 1]
            total_points[idx, 2] = users_height[user]
            idx += 1

    return total_points

# 快速验证：生成一次轨迹看形状
test_traj = generate_human_trajectory()
print(f'Trajectory shape: {test_traj.shape}  (expect {N_TOTAL_POINTS}, 3)')
print(f'X range: [{test_traj[:,0].min():.2f}, {test_traj[:,0].max():.2f}]')
print(f'Y range: [{test_traj[:,1].min():.2f}, {test_traj[:,1].max():.2f}]')

# ==========================================
# 🔮 3. 等效远场衍射信道 — 对齐 MATLAB equivalent_farfield_channel_2
#    nlos_params 参数支持 Monte Carlo 采样
# ==========================================
def equivalent_farfield_channel_pytorch(window_params, locs, nlos_params=None):
    """
    locs: [N, 3] tensor — 轨迹点坐标
    nlos_params: tuple (psi_2, tt_2, eta_2) — 预生成的 NLoS 路径2 参数
                 若为 None 则使用全局预生成参数（向后兼容）
    """
    if not isinstance(window_params, torch.Tensor):
        window_params = torch.tensor(window_params, dtype=torch.float32, device=device)
    xc, zc, Lx, Lz = window_params[0], window_params[1], window_params[2], window_params[3]
    xu, yu, zu = locs[:, 0], locs[:, 1], locs[:, 2]
    N = locs.shape[0]

    dx_BS = xc - x_BS; dy_BS = torch.tensor(0.0 - y_BS, device=device); dz_BS = zc - z_BS
    R_BW = torch.sqrt(dx_BS**2 + dy_BS**2 + dz_BS**2)
    theta_BW = torch.atan2(dy_BS, dx_BS); phi_BW = torch.acos(dz_BS / R_BW)
    k_tx = torch.sin(phi_BW) * torch.cos(theta_BW); k_tz = torch.cos(phi_BW)

    x1, z1 = xc - Lx/2, zc - Lz/2; x2, z2 = xc - Lx/2, zc + Lz/2
    x3, z3 = xc + Lx/2, zc - Lz/2; x4, z4 = xc + Lx/2, zc + Lz/2

    def ray_dir_py(xs, zs):
        dx = xs - x_BS; dy = torch.tensor(0.0 - y_BS, device=device); dz = zs - z_BS
        L = torch.sqrt(dx**2 + dy**2 + dz**2)
        return dx/L, dy/L, dz/L

    ux1, _, uz1 = ray_dir_py(x1, z1); ux2, _, uz2 = ray_dir_py(x2, z2)
    ux3, _, uz3 = ray_dir_py(x3, z3); ux4, _, uz4 = ray_dir_py(x4, z4)

    dx_WU = xu - x_BS; dy_WU = yu - y_BS; dz_WU = zu - z_BS
    L_USER = torch.sqrt(dx_WU**2 + dy_WU**2 + dz_WU**2)
    ux_user = dx_WU / L_USER; uz_user = dz_WU / L_USER

    ux_min = torch.min(torch.stack([ux1, ux2, ux3, ux4]))
    ux_max = torch.max(torch.stack([ux1, ux2, ux3, ux4]))
    uz_min = torch.min(torch.stack([uz1, uz2, uz3, uz4]))
    uz_max = torch.max(torch.stack([uz1, uz2, uz3, uz4]))

    beta_illum = 1000.0
    inx = torch.sigmoid(beta_illum * (ux_user - ux_min)) * torch.sigmoid(beta_illum * (ux_max - ux_user))
    inz = torch.sigmoid(beta_illum * (uz_user - uz_min)) * torch.sigmoid(beta_illum * (uz_max - uz_user))
    in_illumination = inx * inz

    dx_WU2 = xu - xc; dy_WU2 = yu; dz_WU2 = zu - zc
    R_WU = torch.sqrt(dx_WU2**2 + dy_WU2**2 + dz_WU2**2)
    t1 = dx_WU2 / R_WU; t2 = dz_WU2 / R_WU

    ax = (1.0 - in_illumination) * (k_tx - t1)
    az = (1.0 - in_illumination) * (k_tz - t2)

    sincx = torch.sinc((math.pi / lambda_val) * Lx * ax)
    sincz = torch.sinc((math.pi / lambda_val) * Lz * az)

    # ---- 阵列响应 & 多径 ----
    n_ant = torch.arange(E, dtype=torch.float32, device=device)  # n = 0:E-1

    # 路径 1 (LoS) — tt = theta_BW
    phase1 = (2.0 * math.pi / lambda_val) * d_B * n_ant * torch.sin(theta_BW)
    a1 = (1.0 / math.sqrt(E)) * torch.complex(torch.cos(phase1), torch.sin(phase1))
    v1_mag = torch.tensor(lambda_val / (4.0 * math.pi), device=device) / R_BW
    v1_phase = torch.tensor(- (2.0 * math.pi / lambda_val), device=device) * R_BW
    v1 = torch.complex(v1_mag * torch.cos(v1_phase), v1_mag * torch.sin(v1_phase))
    H_path1 = v1 * a1.conj()  # [E]

    # 路径 2 (NLoS) — 使用传入的 nlos_params 或全局预生成
    if nlos_params is not None:
        psi_2, tt_2, eta_2 = nlos_params
    else:
        # fallback: 使用全局预生成（兼容旧代码）
        psi_2, tt_2, eta_2 = _global_psi_2, _global_tt_2, _global_eta_2

    phase2 = (2.0 * math.pi / lambda_val) * d_B * n_ant.unsqueeze(0) * torch.sin(tt_2).unsqueeze(1)  # [N, E]
    a2 = (1.0 / math.sqrt(E)) * torch.complex(torch.cos(phase2), torch.sin(phase2))
    v2_mag = eta_2 * (lambda_val / (4.0 * math.pi * d_ref))
    v2 = torch.complex(v2_mag * torch.cos(psi_2), v2_mag * torch.sin(psi_2))  # [N]
    H_path2 = v2.unsqueeze(1) * a2.conj()  # [N, E]

    # H = sqrt(E/L1) * (Path1 + Path2)
    H_total = math.sqrt(E / L1) * (H_path1.unsqueeze(0) + H_path2)  # [N, E]

    # 窗口衍射因子
    factor_mag = (Lx * Lz * sincx * sincz) / (lambda_val * R_WU)
    factor_phase = torch.tensor(- (2.0 * math.pi / lambda_val), device=device) * (k_tx * xc + k_tz * zc) - (math.pi / 2.0)
    factor = torch.complex(factor_mag * torch.cos(factor_phase), factor_mag * torch.sin(factor_phase))

    H_eq = H_total * factor.unsqueeze(1)  # [N, E]
    return H_eq

# 预生成一组 NLoS 全局默认参数（用于非 MC 场景）
np.random.seed(42)
_global_psi_2 = torch.tensor(2 * np.pi * np.random.rand(N_TOTAL_POINTS), dtype=torch.float32, device=device)
_global_tt_2  = torch.tensor(-np.pi + 2 * np.pi * np.random.rand(N_TOTAL_POINTS), dtype=torch.float32, device=device)
_global_eta_2 = torch.tensor(10 ** ((-15 + 5 * np.random.rand(N_TOTAL_POINTS)) / 10), dtype=torch.float32, device=device)

# ==========================================
# 📊 4. Monte Carlo 目标函数
# ==========================================
penalty_weight = 500.0   # 对齐 MATLAB OUTAGE_PENALTY
target_outage = 0.095    # 0.5% 安全余量，确保 MC100 < 10%

def compute_outage_for_locs(theta_tensor, locs_tensor, nlos_psi, nlos_tt, nlos_eta):
    """对一组轨迹点计算 outage（内部函数）"""
    H_eq = equivalent_farfield_channel_pytorch(
        theta_tensor, locs_tensor,
        nlos_params=(nlos_psi, nlos_tt, nlos_eta))
    H_w = torch.sum(H_eq, dim=1) / math.sqrt(E)
    dp = (torch.abs(H_w) ** 2) * p_m * P_BS
    intf = (n_users - 1) * dp
    sinr = dp / (intf + N0)
    rate = torch.log2(1.0 + sinr)
    outage = torch.mean((rate < R_th).float()).item()
    return outage

def differentiable_loss_mc(theta_input, N_MC=10, hard_mode=True, seed=None):
    """
    Monte Carlo 目标函数：
    生成 N_MC 组独立轨迹 + NLoS 参数，计算平均 outage
    """
    if seed is not None:
        np.random.seed(seed)

    if not isinstance(theta_input, torch.Tensor):
        theta_tensor = torch.tensor(theta_input, dtype=torch.float32, device=device)
    else:
        theta_tensor = theta_input

    xc, zc, Lx, Lz = theta_tensor[0], theta_tensor[1], theta_tensor[2], theta_tensor[3]
    area = Lx * Lz

    # 生成 N_MC 组轨迹并批量计算 outage
    all_outages = []
    for mc in range(N_MC):
        traj = generate_human_trajectory()  # [150, 3]
        locs_t = torch.tensor(traj, dtype=torch.float32, device=device)

        psi_t  = torch.tensor(2 * np.pi * np.random.rand(N_TOTAL_POINTS), dtype=torch.float32, device=device)
        tt_t   = torch.tensor(-np.pi + 2 * np.pi * np.random.rand(N_TOTAL_POINTS), dtype=torch.float32, device=device)
        eta_t  = torch.tensor(10 ** ((-15 + 5 * np.random.rand(N_TOTAL_POINTS)) / 10), dtype=torch.float32, device=device)

        outage = compute_outage_for_locs(theta_tensor, locs_t, psi_t, tt_t, eta_t)
        all_outages.append(outage)

    mean_outage = float(np.mean(all_outages))

    if hard_mode:
        if mean_outage > target_outage:
            cost = area + penalty_weight * (mean_outage - target_outage)
        else:
            cost = area

        #  确保 cost 和 area 是纯 Python 标量，而不是 Tensor
        if isinstance(cost, torch.Tensor):
            cost = cost.detach().cpu().item()

        return cost, mean_outage, all_outages
    else:
        # 可微模式（用于 Adam 精调）：用批量轨迹做可微 loss
        # 这里简化：生成一批轨迹，concat 在一起用 sigmoid 逼近
        all_trajs = []
        all_psis = []; all_tts = []; all_etas = []
        for mc in range(N_MC):
            traj = generate_human_trajectory()
            all_trajs.append(traj)
            all_psis.append(2 * np.pi * np.random.rand(N_TOTAL_POINTS))
            all_tts.append(-np.pi + 2 * np.pi * np.random.rand(N_TOTAL_POINTS))
            all_etas.append(10 ** ((-15 + 5 * np.random.rand(N_TOTAL_POINTS)) / 10))

        big_traj = torch.tensor(np.concatenate(all_trajs, axis=0), dtype=torch.float32, device=device)
        big_psi  = torch.tensor(np.concatenate(all_psis), dtype=torch.float32, device=device)
        big_tt   = torch.tensor(np.concatenate(all_tts), dtype=torch.float32, device=device)
        big_eta  = torch.tensor(np.concatenate(all_etas), dtype=torch.float32, device=device)

        H_eq = equivalent_farfield_channel_pytorch(
            theta_tensor, big_traj, nlos_params=(big_psi, big_tt, big_eta))
        H_w = torch.sum(H_eq, dim=1) / math.sqrt(E)
        dp = (torch.abs(H_w) ** 2) * p_m * P_BS
        intf = (n_users - 1) * dp
        sinr = dp / (intf + N0)
        rate = torch.log2(1.0 + sinr)

        alpha_sigmoid = 300.0
        smooth_outage = torch.mean(torch.sigmoid(alpha_sigmoid * (R_th - rate)))
        penalty_term = torch.relu(smooth_outage - target_outage) * penalty_weight
        return area + penalty_term, mean_outage, all_outages

# ==========================================
# GA 包装：随机 N_MC=3 评估，对齐 MATLAB 的随机-per-eval 逻辑
#         3 组平均 → 降噪但保留探索性，天然筛选鲁棒解
# ==========================================
_ga_eval_count = [0]
_mc_counter = [0]

def ga_obj_wrapper(x):
    _ga_eval_count[0] += 1
    if _ga_eval_count[0] % 500 == 0:
        print(f'    ...{_ga_eval_count[0]} evals', flush=True)
    _mc_counter[0] += 1
    cost, _, _ = differentiable_loss_mc(x, N_MC=3, hard_mode=True,
                                         seed=20000 + _mc_counter[0])
    total_cost = cost + boundary_penalty(x) * 500.0

    # ====== ✨ 修改这里：确保返回给变异/交叉等 GA 算子的是纯 float ======
    if isinstance(total_cost, torch.Tensor):
        total_cost = total_cost.detach().cpu().item()
    return float(total_cost)

# ==========================================
# nonlcon 约束函数 (供参考，GA 改用惩罚方式以加速)
# ==========================================
def boundary_penalty(x):
    xc, zc, Lx, Lz = x[0], x[1], x[2], x[3]
    v = max(0, Lx/2 - xc) + max(0, xc + Lx/2 - room_x) + \
        max(0, Lz/2 - zc) + max(0, zc + Lz/2 - room_z_max)
    return v

# ==========================================
# 🚀 5. 多次独立 GA 运行 + Adam 精调
# ==========================================
N_GA_RUNS = 10
all_ga_results = []

print("="*60)
print(f" 启动 {N_GA_RUNS} 轮独立 GA 全局搜索 (随机 N_MC=3, target outage<{target_outage*100:.1f}%)")
print(f" 每轮: pop=50, gen=100, lb/ub/nonlcon 对齐 MATLAB")
print("="*60)

for run_idx in range(N_GA_RUNS):
    run_seed = 20000 + run_idx * 1000
    _ga_eval_count[0] = 0
    _mc_counter[0] = run_idx * 10000  # 每轮独立随机序列

    print(f"\n--- Run {run_idx+1}/{N_GA_RUNS} (seed={run_seed}) ---")
    t0 = time.time()

    ga = GA(func=ga_obj_wrapper, n_dim=4, size_pop=50, max_iter=150,
            lb=[0.2, 0.2, 0.1, 0.1],
            ub=[9.8, 2.8, 9.8, 2.8])

    best_x_ga, best_y_ga = ga.run()
    elapsed = time.time() - t0

    # 后验大样本评估 (N_MC=50, 独立种子 — 无偏估计)
    _, final_outage, _ = differentiable_loss_mc(best_x_ga, N_MC=50,
                                                  hard_mode=True, seed=90000 + run_idx)
    area_ga = best_x_ga[2] * best_x_ga[3]

    status = '✅' if final_outage <= target_outage else ('⚠️' if final_outage <= 0.11 else '❌')
    print(f"  {status} GA 面积={area_ga:.4f} m² | MC50 outage={final_outage*100:.2f}% | 耗时={elapsed:.1f}s")
    all_ga_results.append({
        'x': best_x_ga.copy(),
        'area': area_ga,
        'outage': final_outage,
        'seed': run_seed
    })

# 选最优（面积最小且 MC50 outage < 9.5%）
valid_results = [r for r in all_ga_results if r['outage'] < target_outage]
if not valid_results:
    # 放宽至 < 11% 并警告
    valid_results = [r for r in all_ga_results if r['outage'] < 0.11]
    print(f"\n⚠️ 无解满足 outage<9.5%, 放宽至 <11%")
best_result = min(valid_results, key=lambda r: r['area'])

print(f"\n{'='*60}")
print(f" 🏆 {N_GA_RUNS} 轮 GA 中最优解: area={best_result['area']:.4f} m², outage={best_result['outage']*100:.2f}%")
print(f"{'='*60}")

# ==========================================
# 🔥 第二阶段：PyTorch Adam 精细调控
# ==========================================
print("\n=====================================================")
print(" ⚡ 第二阶段：PyTorch Adam 精准精调 (MC N=20)...")
print("=====================================================")

refined_theta = torch.tensor(best_result['x'], dtype=torch.float32, device=device, requires_grad=True)
optimizer = torch.optim.Adam([refined_theta], lr=0.001)

for step in range(50):
    optimizer.zero_grad()
    # 可微模式，MC=20
    # 固定 seed: Adam 在确定的 loss landscape 上精调
    loss, _, _ = differentiable_loss_mc(refined_theta, N_MC=20,
                                         hard_mode=False, seed=85000)
    loss.backward()

    with torch.no_grad():
        refined_theta[2].clamp_(0.1, room_x)
        refined_theta[3].clamp_(0.1, room_z_max)
        refined_theta[0].clamp_(refined_theta[2]/2, room_x - refined_theta[2]/2)
        refined_theta[1].clamp_(refined_theta[3]/2, room_z_max - refined_theta[3]/2)

    optimizer.step()
    if step % 10 == 0:
        print(f"  Adam step {step:02d} | loss={loss.item():.4f}")

final_res = refined_theta.detach().cpu().numpy()
_, final_mc_outage, _ = differentiable_loss_mc(final_res, N_MC=100,
                                                 hard_mode=True, seed=99999)
area_pytorch_final = final_res[2] * final_res[3]

# ==========================================
# 📊 成果看板
# ==========================================
print("\n" + "="*72)
print("🏆 电磁窗口优化：Monte Carlo GA(对齐MATLAB) + PyTorch 微分精调")
print("="*72)
print(f" 项目指标         | 阶段一(MC-GA最优)    | 阶段二(GA+Adam精调)")
print("-"*72)
print(f" 窗口中心 xc     |      {best_result['x'][0]:7.3f} m       |       {final_res[0]:7.3f} m")
print(f" 窗口中心 zc     |      {best_result['x'][1]:7.3f} m       |       {final_res[1]:7.3f} m")
print(f" 窗口长度 Lx     |      {best_result['x'][2]:7.3f} m       |       {final_res[2]:7.3f} m")
print(f" 窗口高度 Lz     |      {best_result['x'][3]:7.3f} m       |       {final_res[3]:7.3f} m")
print("-"*72)
print(f" 中断概率(MC100) |      {best_result['outage']*100:7.2f} %       |       {final_mc_outage*100:7.2f} %")
print(f" 部署面积        |      {best_result['area']:7.4f} m²       |       {area_pytorch_final:7.4f} m²")
print("="*72)
area_dist = [f"{r['area']:.4f}" for r in all_ga_results]
print(f"\n 10轮GA面积分布: {area_dist}")
print(f" MATLAB 基准: area=0.0120 m², outage=9.33% (注意:随机目标单样本)")
print("="*72)